### Lab 5.1: a search_orders tool, backed by a mock database, wired into Lab 3.1's hand-rolled tool-use loop.


Run from the PARENT of ccdv_f_labs/:
    python -m ccdv_f_labs.lab_5_1_search_orders_tool


In [ ]:
import sys
from pathlib import Path
import anthropic
import time
import gradio as gr

sys.path.insert(0, str(Path.cwd().parent))

from common import get_client, MODEL

client = get_client()

# --- Mock database: a few orders spread across a few customers ---
ORDERS_DB = [
    {"order_id": "ORD-001", "customer_id": "CUST-100", "status": "shipped"},
    {"order_id": "ORD-002", "customer_id": "CUST-100", "status": "pending"},
    {"order_id": "ORD-003", "customer_id": "CUST-200", "status": "shipped"},
    {"order_id": "ORD-004", "customer_id": "CUST-200", "status": "cancelled"},
    {"order_id": "ORD-005", "customer_id": "CUST-300", "status": "pending"},
]

# --- Tool schema: customer_id (string, required), status (enum, optional) ---
TOOLS = [
    {
        "name": "search_orders",
        "description": (
            "Search the order database for a customer's orders. Filters by "
            "customer_id, and optionally narrows further by status."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "The customer's ID, e.g. 'CUST-100'.",
                },
                "status": {
                    "type": "string",
                    "enum": ["pending", "shipped", "cancelled"],
                    "description": "Optional: only return orders with this status.",
                },
            },
            "required": ["customer_id"],
        },
    },
]


# --- The function actually backing the tool -- hits the mock "database" ---
def _search_orders(customer_id: str, status: str | None = None) -> str:
    # Defensive check even though `status` is schema-constrained to an enum --
    # never trust model output to have respected the schema perfectly.
    valid_statuses = {"pending", "shipped", "cancelled"}
    if status is not None and status not in valid_statuses:
        return f"Invalid status '{status}'. Must be one of {sorted(valid_statuses)}."

    matches = [
        o for o in ORDERS_DB
        if o["customer_id"] == customer_id and (status is None or o["status"] == status)
    ]
    if not matches:
        scope = f" with status '{status}'" if status else ""
        return f"No orders found for {customer_id}{scope}."
    return "\n".join(f"{o['order_id']}: {o['status']}" for o in matches)


TOOL_IMPLS = {"search_orders": _search_orders}


# --- Lab 3.1's loop, unchanged in shape, now wired to this tool ---
def run_agent(task: str, max_iterations: int = 5) -> str:
    messages = [{"role": "user", "content": task}]
    for _ in range(max_iterations):
        resp = client.messages.create(
            model=MODEL, max_tokens=500, tools=TOOLS, messages=messages,
        )
        messages.append({"role": "assistant", "content": resp.content})

        if resp.stop_reason != "tool_use":
            text_block = next(b for b in resp.content if b.type == "text")
            return text_block.text

        tool_results = []
        for block in resp.content:
            if block.type != "tool_use":
                continue
            impl = TOOL_IMPLS.get(block.name)
            result = impl(**block.input) if impl else f"Unknown tool: {block.name}"
            tool_results.append(
                {"type": "tool_result", "tool_use_id": block.id, "content": result}
            )
        messages.append({"role": "user", "content": tool_results})

    return "Agent hit max_iterations without finishing."


if __name__ == "__main__":
    answer = run_agent(
        "Does customer CUST-200 have any cancelled orders? If so, which ones?"
    )
    print(answer)